# African Bat Specimen Records Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 African Bat Specimen Records dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) accessible by URL.

In [ ]:
# Install the `mlcroissant` library if not already installed
!pip install mlcroissant pandas --quiet

## 1. Data Loading
Load the dataset metadata and explore the information structure with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.p39y-mhbj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded!")
print(f"Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Explore available record sets, their `@id`s, and the fields they provide.

In [ ]:
from pprint import pprint

# List all available record sets
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecord Set: Name='{rs.name}', @id='{rs.id}'")
    # Display the fields in this record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id='{field.id}', type={field.data_type})")

## 3. Data Extraction
Load records from a chosen record set into a DataFrame for exploration. All references use the `@id` attributes as shown above.

In [ ]:
# Identify primary record set (specimens dataset) by its @id.
# Inspect the record set IDs from the previous overview step if needed.
main_record_set_id = None
for rs in record_sets:
    # Heuristically pick the primary record set which contains core specimen records
    if 'specimen' in rs.name.lower() or 'bat' in rs.name.lower() or 'record' in rs.name.lower():
        main_record_set_id = rs.id
        break

if main_record_set_id is None and record_sets:
    # Otherwise fall back to the first available
    main_record_set_id = record_sets[0].id

print(f"Main record set @id: {main_record_set_id}")

# Load data from all record sets
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} rows for record set: {rs.name} (@id='{rs.id}')")

if main_record_set_id in dataframes:
    print("\nMain data columns:")
    pprint(list(dataframes[main_record_set_id].columns))
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate filtering, normalization, and grouping. We reference fields by their `@id` (as column names in DataFrame).
- For example, analyze records by `year` (@id may differ; check above data columns), filter for recent years, and normalize specimen years.
- Group by country if available.

In [ ]:
df = dataframes[main_record_set_id]
# Typical field ids may be 'schema:year', 'schema:country', etc. Confirm column names
# Try to find year and country fields
year_field = None
country_field = None

for col in df.columns:
    if 'year' in col.lower():
        year_field = col
    if 'country' in col.lower():
        country_field = col

if year_field is None:
    print('No year field found.')
else:
    # Ensure numeric
    df[year_field] = pd.to_numeric(df[year_field], errors='coerce')
    threshold = 1990

    filtered_df = df[df[year_field] > threshold]
    print(f"Filtered to records where {year_field} > {threshold} ({len(filtered_df)} records):")
    display(filtered_df.head())

    filtered_df[f"{year_field}_normalized"] = (filtered_df[year_field] - filtered_df[year_field].mean()) / filtered_df[year_field].std()
    print(f"\nNormalized {year_field} for filtered records:")
    display(filtered_df[[year_field, f"{year_field}_normalized"]].head())

    # Group by country if available
    if country_field and country_field in filtered_df.columns:
        print(f"\nNumber of records per country with {year_field} > {threshold}:")
        grouped = filtered_df.groupby(country_field)[year_field].count().rename('record_count')
        display(grouped.head())

## 5. Visualization
Let's plot the distribution of specimen years and records by country.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

if year_field in df.columns:
    plt.figure(figsize=(9,4))
    df[year_field].dropna().astype(int).hist(bins=30)
    plt.title('Distribution of Specimen Collection Years')
    plt.xlabel('Year')
    plt.ylabel('Count')
    plt.show()

if country_field and country_field in df.columns:
    top_countries = df[country_field].value_counts().head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=top_countries.index, y=top_countries.values)
    plt.title('Top 10 Countries by Number of Records')
    plt.ylabel('Count')
    plt.xlabel('Country')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We loaded the African Bat Specimen Records dataset using the `mlcroissant` library, inspected its record sets, and performed simple exploratory analysis. 

- The main record set contains rows with years, countries, and taxonomic information.
- We filtered recent records, normalized year values, and visualized data distributions.

**You can use the `mlcroissant` library to load other Croissant datasets and build analyses reproducibly using only public schema URLs and referencing fields by `@id`.**